# DQN baseline on Four Rooms

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.fourrooms_discrete import FourRoomsGridWorld, FourRoomsGoalWrapper
from utils import TrajectoryReplayBufferDiscrete, evaluate_policy
from visualisations import plot_policy_rollouts, plot_q_diagnostics
from networks import DQN_QNetwork


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
def make_env(goal=(9, 9), slip_prob=0.00, max_horizon=500):
    base = FourRoomsGridWorld(room_size=5, max_episode_steps=max_horizon)
    env = FourRoomsGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
    )
    return env


In [ ]:

def dqn_train(
    total_steps=100000,
    warmup_steps=5000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    lr=1e-3,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_steps=50000,
    buffer_capacity=100000,
    train_freq=4,
):
    env = make_env()
    obs_dim = env.observation_space.shape[0]
    num_actions = env.action_space.n

    q_net = DQN_QNetwork(obs_dim, num_actions).to(DEVICE)
    q_target = DQN_QNetwork(obs_dim, num_actions).to(DEVICE)
    q_target.load_state_dict(q_net.state_dict())
    for p in q_target.parameters():
        p.requires_grad_(False)

    opt = optim.Adam(q_net.parameters(), lr=lr)
    buffer = TrajectoryReplayBufferDiscrete(buffer_capacity, obs_dim, 1, device=DEVICE)

    obs, _ = env.reset()
    global_step = 0
    eval_returns = []

    ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = [], [], [], [], [], []

    while global_step < total_steps:
        # Epsilon-greedy action selection
        frac = min(1.0, global_step / eps_decay_steps)
        eps = eps_start + frac * (eps_end - eps_start)

        if np.random.random() < eps:
            action = env.action_space.sample()
        else:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                action = int(q_net(obs_t).argmax(dim=-1).item())

        next_obs, rew, term, trunc, _ = env.step(action)
        done = term or trunc

        ep_obs.append(obs.copy())
        ep_actions.append(action)
        ep_rewards.append(float(rew))
        ep_next_obs.append(next_obs.copy())
        ep_terminated.append(float(term))
        ep_truncated.append(float(trunc))

        obs = next_obs
        global_step += 1

        if done:
            episode = {
                'obs': ep_obs,
                'actions': ep_actions,
                'rewards': ep_rewards,
                'next_obs': ep_next_obs,
                'terminated': ep_terminated,
                'truncated': ep_truncated,
            }
            buffer.add_episode(episode)
            ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = [], [], [], [], [], []
            obs, _ = env.reset()

        # Training update
        if len(buffer) >= warmup_steps and global_step % train_freq == 0:
            batch = buffer.sample(batch_size)

            obs_t = batch.obs
            act_t = batch.actions.long()
            rew_t = batch.rewards
            next_obs_t = batch.next_obs
            term_t = batch.terminated

            with torch.no_grad():
                next_q = q_target(next_obs_t).max(dim=-1, keepdim=True).values
                target = rew_t + gamma * (1.0 - term_t) * next_q

            current_q = q_net(obs_t).gather(1, act_t.unsqueeze(1))
            loss = F.mse_loss(current_q, target)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
            opt.step()

            # Soft update target network
            for p, p_tgt in zip(q_net.parameters(), q_target.parameters()):
                p_tgt.data.mul_(1.0 - tau).add_(tau * p.data)

        if global_step % 10000 == 0:
            eval_env = make_env()
            mean_ret, mean_len = evaluate_policy(
                eval_env,
                lambda o: int(
                    q_net(
                        torch.tensor(o, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                    ).argmax(dim=-1).item()
                ),
                episodes=8,
            )
            eval_returns.append((global_step, mean_ret))
            print(f'[DQN] step={global_step:7d} | eps={eps:.3f} | eval_return={mean_ret:.3f} | eval_len={mean_len:.1f}')
            eval_env.close()

    env.close()
    return q_net, eval_returns


dqn_q, dqn_eval = dqn_train()

if dqn_eval:
    xs, ys = zip(*dqn_eval)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel('Environment steps')
    plt.ylabel('Mean episodic return')
    plt.title('DQN baseline on FourRooms (discrete)')
    plt.grid(alpha=0.25)
    plt.show()


## Visualisations

In [ ]:

dqn_q.eval()
eval_env = make_env(goal=(9, 9))


def dqn_policy_fn(obs):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        action = int(dqn_q(obs_t).argmax(dim=-1).item())
    return action


def dqn_value_fn(obs_batch):
    obs_t = torch.tensor(obs_batch, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        q_vals = dqn_q(obs_t).cpu().numpy()
    return q_vals


plot_policy_rollouts(
    env=eval_env,
    policy_fn=dqn_policy_fn,
    goal_pos=(9, 9),
    eval_episodes=8,
    n_cols=4,
    is_discrete=True,
    step_point_size=12,
    start_size=60,
    end_size=50,
    goal_size=130,
    arrow_width=0.01,
)

eval_env.close()

eval_env = make_env(goal=(9, 9))

plot_q_diagnostics(
    env=eval_env,
    value_fn=dqn_value_fn,
    actor_fn=None,
    is_discrete=True,
    num_actions=4,
    action_names=['Up', 'Down', 'Left', 'Right'],
    goal_pos=(9, 9),
    eval_returns=dqn_eval,
)

eval_env.close()
